In [2]:
import pandas as pd

In [3]:
# Load the diabetes dataset
df = pd.read_csv("diabetic_data.csv")

# Display the first five rows
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [12]:
# Select relevant variables for this assignment
df = df[[
    "encounter_id",
    "age",
    "gender",
    "time_in_hospital",
    "num_lab_procedures",
    "num_medications",
    "readmitted"
]]
df.head()

,encounter_id,age,gender,time_in_hospital,num_lab_procedures,num_medications,readmitted
0,2278392,[0-10),Female,1,41,1,NO
1,149190,[10-20),Female,3,59,18,>30
2,64410,[20-30),Female,2,11,13,NO
3,500364,[30-40),Male,2,44,16,NO
4,16680,[40-50),Male,1,51,8,NO


In [13]:
# 1. MELT: Convert wide format to long format

melted_df = pd.melt(
    df.head(5),
    id_vars=["encounter_id"],
    value_vars=["time_in_hospital",
                "num_lab_procedures",
                "num_medications"],
    var_name="measurement",
    value_name="value"
)
melted_df

,encounter_id,measurement,value
0,2278392,time_in_hospital,1
1,149190,time_in_hospital,3
2,64410,time_in_hospital,2
3,500364,time_in_hospital,2
4,16680,time_in_hospital,1
5,2278392,num_lab_procedures,41
6,149190,num_lab_procedures,59
7,64410,num_lab_procedures,11
8,500364,num_lab_procedures,44
9,16680,num_lab_procedures,51


**Melt:** The `pd.melt()` function transforms a DataFrame from wide format to long format. In this example, I selected five hospital encounters and converted three variables (time in hospital, number of lab procedures, and number of medications) into two columns: measurement and value. This transformation makes the data easier to organize and analyze by measurement type.

In [14]:
# 2. PIVOT: Convert long format to wide format

pivoted_df = melted_df.pivot(
    index="encounter_id",
    columns="measurement",
    values="value"
)
pivoted_df

measurement,num_lab_procedures,num_medications,time_in_hospital
encounter_id,,,
16680,51,8,1
64410,11,13,2
149190,59,18,3
500364,44,16,2
2278392,41,1,1


**Pivot:** The `pivot()` function reshapes data from long format to wide format. In this example, I used the melted dataset to reorganize the measurements into separate columns for each hospital encounter. The result displays the number of lab procedures, number of medications, and time in hospital as individual columns.

In [16]:
# 3. AGGREGATION: Calculate summary statistics

aggregation_df = df[
    ["time_in_hospital", "num_lab_procedures", "num_medications"]
].agg(["mean", "min", "max"])

aggregation_df

,time_in_hospital,num_lab_procedures,num_medications
mean,4.395987,43.095641,16.021844
min,1.000000,1.000000,1.000000
max,14.000000,132.000000,81.000000


**Aggregation:** The `agg()` function calculates summary statistics for numerical variables in a DataFrame. In this example, I used `mean`, `min`, and `max` to analyze the time spent in the hospital, the number of laboratory procedures, and the number of medications. This allows us to summarize and better understand the characteristics of hospital encounters in the diabetes dataset.

In [17]:
# 4. ITERATION: Loop through the first five hospital encounters

for index, row in df.head(5).iterrows():
    print("Encounter ID:", row["encounter_id"])
    print("Age group:", row["age"])
    print("Days in hospital:", row["time_in_hospital"])
    print("--------------------")

Encounter ID: 2278392
Age group: [0-10)
Days in hospital: 1
--------------------
Encounter ID: 149190
Age group: [10-20)
Days in hospital: 3
--------------------
Encounter ID: 64410
Age group: [20-30)
Days in hospital: 2
--------------------
Encounter ID: 500364
Age group: [30-40)
Days in hospital: 2
--------------------
Encounter ID: 16680
Age group: [40-50)
Days in hospital: 1
--------------------


**Iteration:** The `iterrows()` function allows us to iterate through the rows of a DataFrame. In this example, I used a `for` loop to examine the first five hospital encounters and display the encounter ID, patient age group, and number of days spent in the hospital. This demonstrates how to access and process individual records in a dataset.

In [18]:
# 5. GROUPBY: Average hospital stay by readmission category

grouped_df = df.groupby("readmitted")["time_in_hospital"].mean()

grouped_df

readmitted
<30    4.768249
>30    4.495541
NO     4.254429
Name: time_in_hospital, dtype: float64

**Groupby:** The `groupby()` function organizes data into groups based on a specific variable. In this example, I grouped hospital encounters by readmission category and calculated the average length of hospital stay for each group. The results showed that encounters followed by readmission within 30 days had the longest average hospital stay (4.77 days), followed by readmissions after 30 days (4.50 days) and encounters with no recorded readmission (4.25 days). These results suggest a descriptive association between hospital stay duration and readmission status, but they do not establish causation.